# Fase 0 — Entorno y descarga de Sen1Floods11
Pregunta 2: mapeo de inundaciones con U-Net sobre Sentinel-1.

## 1. Verificar GPU

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 2. Montar Drive y crear la estructura del proyecto (el "repositorio")

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
RAIZ_PROYECTO = Path('/content/drive/MyDrive/rnap_p2_inundaciones')
for sub in ['src', 'notebooks', 'resultados', 'checkpoints', 'figuras', 'informe']:
    (RAIZ_PROYECTO / sub).mkdir(parents=True, exist_ok=True)
print('Estructura lista en', RAIZ_PROYECTO)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Estructura lista en /content/drive/MyDrive/rnap_p2_inundaciones


**Ahora sube** `configuracion.py`, `utilidades.py` y `__init__.py` a `rnap_p2_inundaciones/src/` en tu Drive (y este notebook a `notebooks/`). Luego corre la siguiente celda.

In [3]:
import sys
sys.path.insert(0, str(RAIZ_PROYECTO))

from src.configuracion import *
from src.utilidades import fijar_semilla, obtener_dispositivo

fijar_semilla(SEMILLA)
dispositivo = obtener_dispositivo()

GPU: Tesla T4


**Celda de Diagnostico**

In [6]:
!ls -la /content/drive/MyDrive/rnap_p2_inundaciones/
!ls -la /content/drive/MyDrive/rnap_p2_inundaciones/src/

total 32
drwx------ 8 root root 4096 Oct  4 17:14 .
drwx------ 5 root root 4096 Oct  4 17:14 ..
drwx------ 2 root root 4096 Oct  4 17:14 checkpoints
drwx------ 2 root root 4096 Oct  4 17:14 figuras
drwx------ 2 root root 4096 Oct  4 17:14 informe
drwx------ 2 root root 4096 Oct  4 17:14 notebooks
drwx------ 2 root root 4096 Oct  4 17:14 resultados
drwx------ 3 root root 4096 Oct  4 17:15 src
total 13
drwx------ 3 root root 4096 Oct  4 17:15 .
drwx------ 8 root root 4096 Oct  4 17:14 ..
-rw------- 1 root root    0 Oct  4 17:08 __init__.py
drwx------ 2 root root 4096 Oct  4 17:15 __pycache__
-rw------- 1 root root  973 Oct  4 17:08 utilidades.py


## 3. Librerías geoespaciales
`rasterio` lee GeoTIFF (imagen + georreferencia). Colab no la trae por defecto.

In [4]:
!pip install -q rasterio

## 4. Descargar solo lo etiquetado a mano + particiones
`-m` = descarga en paralelo. El bucket es público, no pide credenciales.

In [5]:
rutas_descarga = {
    'data/flood_events/HandLabeled/S1Hand':          DIR_S1_MANO.parent,
    'data/flood_events/HandLabeled/LabelHand':       DIR_ETIQUETA_MANO.parent,
    'data/flood_events/HandLabeled/S1OtsuLabelHand': DIR_OTSU_MANO.parent,
    'splits/flood_handlabeled':                      DIR_PARTICIONES.parent,
}
for origen, destino in rutas_descarga.items():
    destino.mkdir(parents=True, exist_ok=True)
    !gsutil -m -q cp -r {BUCKET}/{origen} {destino}
print('Descarga terminada')

Descarga terminada


## 5. Verificación

In [6]:
for carpeta in [DIR_S1_MANO, DIR_ETIQUETA_MANO, DIR_OTSU_MANO]:
    print(f'{carpeta.name:18s} -> {len(list(carpeta.glob("*.tif")))} archivos')

print()
for csv in sorted(DIR_PARTICIONES.glob('*.csv')):
    n = sum(1 for _ in open(csv))
    print(f'{csv.name:28s} -> {n} chips')

S1Hand             -> 446 archivos
LabelHand          -> 446 archivos
S1OtsuLabelHand    -> 446 archivos

flood_bolivia_data.csv       -> 15 chips
flood_test_data.csv          -> 90 chips
flood_train_data.csv         -> 252 chips
flood_valid_data.csv         -> 89 chips


In [7]:
import rasterio

ejemplo_s1 = sorted(DIR_S1_MANO.glob('*.tif'))[0]
with rasterio.open(ejemplo_s1) as src:
    print('Archivo :', ejemplo_s1.name)
    print('Bandas  :', src.count, '| Tamaño:', src.width, 'x', src.height, '| Tipo:', src.dtypes)
    print('CRS     :', src.crs, '| Resolución (grados):', src.res)

with open(sorted(DIR_PARTICIONES.glob('*.csv'))[0]) as f:
    print('\nPrimeras filas de un CSV de partición:')
    for _ in range(3): print('  ', f.readline().strip())

Archivo : Bolivia_103757_S1Hand.tif
Bandas  : 2 | Tamaño: 512 x 512 | Tipo: ('float32', 'float32')
CRS     : EPSG:4326 | Resolución (grados): (8.983152841195215e-05, 8.983152841195215e-05)

Primeras filas de un CSV de partición:
   Bolivia_103757_S1Hand.tif,Bolivia_103757_LabelHand.tif
   Bolivia_129334_S1Hand.tif,Bolivia_129334_LabelHand.tif
   Bolivia_195474_S1Hand.tif,Bolivia_195474_LabelHand.tif


## Qué deberías ver
- `S1Hand`, `LabelHand`, `S1OtsuLabelHand`: **446** archivos cada uno.
- CSVs: `flood_train_data` (~252), `flood_valid_data` (~89), `flood_test_data` (~90), `flood_bolivia_data` (~15).
- S1: **2 bandas** (VV, VH), 512×512, `float32`, CRS EPSG:4326.